# Modelling — Split A CV + Split B deployment verdict

**Input:** `features_window.csv` + `window_index.csv`.
**Protocol** (locked in exploration):
- binary healthy vs damaged; **AUROC / PR-AUC** primary (majority ≈ 70 %).
- **Split A** — stratified bearing-wise 5-fold CV; compare logistic regression
  vs gradient boosting, keep the better one on mean per-condition bearing AUROC.
- **Split B** — the deployment verdict, run ONCE: train on artificial + healthy,
  test on real/combination + healthy (healthy = shared reference, the standard
  protocol for this dataset; damaged bearings stay strictly partitioned).

Artifacts: `data/models/model.joblib` + `config.json`, `data/processed/results/*`.

In [ ]:
import pandas as pd

# Train + evaluate both splits, save artifacts
from pipeline.model import run_pipeline

out = run_pipeline()

In [ ]:
# Split A table: window + per-condition bearing AUROC
display(out["split_a"].round(3))

In [ ]:
# Split B verdict — per-condition bearing AUROC / PR-AUC
sb = out["split_b"]
print("window AUROC:", round(sb["window_auroc"], 3), "| window PR:", round(sb["window_pr"], 3))
print("majority baseline (all-damaged):", round(sb["baseline_precision"], 3))
print("\nper-condition bearing AUROC:")
display(pd.Series(sb["cond_auroc"], name="auroc").round(3))
print("per-condition bearing PR-AUC:")
display(pd.Series(sb["cond_aupr"], name="aupr").round(3))

In [ ]:
# Bearing-level verdicts on Split B test (80 = 20 bearings x 4 conditions)
bl = sb["bearing_level"].copy()
bl["verdict"] = (bl["prob"] >= 0.5).map({True: "damaged", False: "healthy"})
display(bl.sort_values("prob", ascending=False).head(12))
print("\nconfusion (rows=label, cols=verdict  [healthy, damaged]):")
print(sb["bearing_confusion"])

## Findings
- **Split A**: logistic regression beats gradient boosting on bearing-level
  AUROC (0.80 vs 0.64) — chosen for deployment.
- **Split B (artificial → real transfer)**: window AUROC **0.78**, window PR **0.85**
  (baseline 0.70); per-condition bearing AUROC 0.77–0.89 (max at low-load F01).
- Honest read: MCSA current features separate healthy vs real damage better than
  chance, but the artificial→real transfer is **moderate, not perfect** — false
  alarms on 8/24 healthy bearing-condition pairs at 0.5 threshold.
- **Next (ops)**: per-bearing dashboard, threshold targeting a required
  recall, and on-stream validation before field deployment.